In [1]:
# qa.txt 파일을 로드 (리스트 형태로 구성)
text = open(
    '../data/qa.txt', encoding='utf-8'
).read()

In [2]:
type(text)

str

In [3]:
# eval() : 문자를 python의 구조로 변경하는 함수
qa_list = eval(text)

In [4]:
type(qa_list)

list

In [5]:
qa_list

[('환불은 어떻게 하나요?', "주문 상세 페이지에서 '환불 신청' 버튼을 눌러 접수하실 수 있습니다."),
 ('배송 기간은 얼마나 걸리나요?', '일반 배송은 2~3일, 도서산간 지역은 최대 5일까지 소요됩니다.'),
 ('해외 배송도 가능한가요?', '현재 해외 배송은 지원하지 않습니다.'),
 ('회원 탈퇴는 어디에서 하나요?', '설정 > 계정 관리 > 회원 탈퇴 메뉴에서 진행하실 수 있습니다.'),
 ('비밀번호를 잊어버렸어요', "로그인 화면의 '비밀번호 재설정' 링크를 통해 재설정 가능합니다."),
 ('주문 취소는 어떻게 하죠?', '상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다.'),
 ('영수증 발급이 가능한가요?', '마이페이지 > 주문 내역에서 영수증 출력이 가능합니다.'),
 ('교환/반품은 가능한가요?', '수령일로부터 7일 이내, 미사용/미훼손 제품에 한해 가능합니다.')]

In [6]:
for data in qa_list:
    print(data[0])

환불은 어떻게 하나요?
배송 기간은 얼마나 걸리나요?
해외 배송도 가능한가요?
회원 탈퇴는 어디에서 하나요?
비밀번호를 잊어버렸어요
주문 취소는 어떻게 하죠?
영수증 발급이 가능한가요?
교환/반품은 가능한가요?


In [7]:
for q,a in qa_list:
    print(q)

환불은 어떻게 하나요?
배송 기간은 얼마나 걸리나요?
해외 배송도 가능한가요?
회원 탈퇴는 어디에서 하나요?
비밀번호를 잊어버렸어요
주문 취소는 어떻게 하죠?
영수증 발급이 가능한가요?
교환/반품은 가능한가요?


In [8]:
questions = [q for q, a in qa_list]
questions

['환불은 어떻게 하나요?',
 '배송 기간은 얼마나 걸리나요?',
 '해외 배송도 가능한가요?',
 '회원 탈퇴는 어디에서 하나요?',
 '비밀번호를 잊어버렸어요',
 '주문 취소는 어떻게 하죠?',
 '영수증 발급이 가능한가요?',
 '교환/반품은 가능한가요?']

In [9]:
# 분석기는 komoran을 사용
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

- 코사인 유사도
    - 문장과 문장 사이에 어느 정도 같은 의미를 가지는가?
    - 고차원 벡터에서 유클리드 거리를 사용하게 되면 차원이 올라갈수록 거리가 멀어지는 현상이 발생
    - 거리가 아닌 각도를 기준으로 유사도를 체크하는 방법 (일반적으로 사용하는 방법)
    - 1 : 같은 의미
    - -1 : 반대 의미
    - 0 : 서로 무관

In [13]:
# 토큰화 -> 백터화 
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vec = TfidfVectorizer(
    tokenizer= tokenize, 
    ngram_range=(1, 2), 
    lowercase=False
)

In [14]:
# 질문 목록을 벡터화 
X = vec.fit_transform(questions)

In [15]:
X.toarray().shape

(8, 76)

In [16]:
# 새로운 질문 생성 
query = "환불을 하려면 어떻게 하면 될까요?"
# 새로운 질문을 벡터화 
query_vec = vec.transform([query])

In [17]:
# 코사인 유사도 -> 어떤 값들을 비교할것인가? -> 새로운 질문의 벡터 데이터와 질문 목록들의 벡터 데이터를 비교
# cosine_similarity(query_vec, X)[0]
# ravel() : array에서 사용하는 함수 .   다차원 행렬을 1차원 행렬로 변경하는 함수 
sims = cosine_similarity(query_vec, X).ravel()

In [18]:
# argsort() : 배열의 값을 정렬 했을때 그 정렬 순서를 되돌려주는 함수 
rank = sims.argsort()[::-1]

In [19]:
print("질문 : ", query)
for i in rank[:2]:
    print(f"index : {i}, 유사도 : {round(sims[i], 3)}  \n 유사 질문 : {questions[i]}, 답변 : {qa_list[i][1]}")

질문 :  환불을 하려면 어떻게 하면 될까요?
index : 0, 유사도 : 0.69  
 유사 질문 : 환불은 어떻게 하나요?, 답변 : 주문 상세 페이지에서 '환불 신청' 버튼을 눌러 접수하실 수 있습니다.
index : 5, 유사도 : 0.453  
 유사 질문 : 주문 취소는 어떻게 하죠?, 답변 : 상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다.
